# Run basic calculations using Workgraph

## Aim

The workgraph demonstrates combining a gemotery optimisation followed by a phonon calculation to obtain force constants. Both employ Janus-core

### Setup

The initial setup is very similar to the other tutorials, such as `singlepoint.ipynb`, which goes into more detail about what each step is doing. For simplicity all the imports are done at the start.

In [10]:
from aiida.plugins import CalculationFactory
from aiida import load_profile
from aiida_mlip.data.model import ModelData
from aiida.orm import StructureData
from aiida.orm import load_code
from aiida.orm import Str, Float, Bool, Int

from aiida_workgraph import WorkGraph

from ase.build import bulk
from ase.io import read

Load the aiida profile:

In [2]:
# Load profile
load_profile()

Profile<uuid='a0bc4bdaef58416f840387aceea091a9' name='john'>

Get the structure, model and load the code. NaCl is being used for the structure (note the lattice parameter).

In [3]:
#FCC NaCl with 5.63 lattice parameter. This gives 2.815 in the reduced unit cell.
structure = StructureData(ase=bulk("NaCl", "rocksalt", 5.63))

In [ ]:
model = ModelData.from_local("/path/to/model", architecture="mace")

#or from url
#uri = "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model"
#model = ModelData.from_uri(uri, architecture="mace", cache_dir="mlips")

In [7]:
code = load_code("janus@localhost")

Inputs should include the model, code, metadata, and any other keyword arguments expected by the calculation we are running. The input for the geometry optimisation is completed first followed by the inputs for the phonon calculation.

In [8]:
#inpits for the geometry optimisation
inputs_geom = {
        "code": code,
        "model": model,
        "struct": structure,
        "arch": Str(model.architecture),
        "device": Str("cpu"),
        "fmax": Float(0.1),
        "opt_cell_lengths": Bool(True),
        "opt_cell_fully": Bool(True),
        "metadata": {"options": {"resources": {"num_machines": 1}}},
}


In [11]:
#input for phonon.
inputs_phon = {
        "metadata": {"options": {"resources": {"num_machines": 1}}},
        "code": code,
        "arch": model.architecture,
        "model": model,
        "device": Str("cpu"),
        "supercell": Str("2 2 2"),
        "minimize": Bool(False),
        "fmax": Float(0.1),
        "displacement": Float(0.01),
        "nqpoints": Int(51),
        "dos": Bool(False),
        "pdos": Bool(False),
        "bands": Bool(False),
        "no_hdf5": Bool(False),
        "symmetrize": Bool(False),
}


Instantiate both geometry optimisation and phonon calculations.

In [12]:
geomoptCalc = CalculationFactory("mlip.opt")
phononCalc = CalculationFactory("mlip.ph")

### Creating and running a workgraph

We can now create a workgraph by first loading WorkGraph and giving it a name  (`"GeomOptPhonGraph"` in this example). 

In [13]:
wg = WorkGraph("GeomOptPhonGraph")



We then create a task for our calculation and assign this task a name. The name is then used to retrieve the final structure for use in the phonon calculation.

In [14]:
gm_calc = wg.add_task(
    geomoptCalc,
    name="geomopt_calc",
    **inputs_geom
)

opt_struct = gm_calc.outputs.final_structure

Create a task for the phonon calculation using the optimised structure.

In [15]:
ph_calc = wg.add_task(
    phononCalc,
    name="ph_calc",
    struct = opt_struct,
    **inputs_phon,
)


For geometry optimisation we are going to pass outputs to the graph

In [16]:
wg.outputs.results = wg.tasks.geomopt_calc.outputs.results_dict
wg.outputs.results_file = wg.tasks.geomopt_calc.outputs.xyz_output

In [17]:
wg

We can finally run the tasks

In [18]:
wg.run()

03/26/2026 10:06:07 AM <57317> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
03/26/2026 10:06:07 AM <57317> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
03/26/2026 10:06:07 AM <57317> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.
03/26/2026 10:06:09 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|continue_workgraph]: tasks ready to run: geomopt_calc
03/26/2026 10:06:09 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|on_wait]: Process status: Waiting for child processes: 4577


the output before exit <class 'aiida_mlip.calculations.singlepoint.Singlepoint'> <class 'aiida_mlip.calculations.geomopt.GeomOpt'>


03/26/2026 10:06:20 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|update_task_state]: Task: geomopt_calc, type: CALCJOB, finished.
03/26/2026 10:06:21 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|continue_workgraph]: tasks ready to run: ph_calc
03/26/2026 10:06:21 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|on_wait]: Process status: Waiting for child processes: 4591
03/26/2026 10:06:32 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|update_task_state]: Task: ph_calc, type: CALCJOB, finished.
03/26/2026 10:06:32 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|continue_workgraph]: tasks ready to run: 
03/26/2026 10:06:32 AM <57317> aiida.orm.nodes.process.workflow.workchain.WorkChainNode: [REPORT] [4573|WorkGraphEngine|fina

{'results': <Dict: uuid: 435780f8-98a9-4d82-83dd-a745f92afb75 (pk: 4583)>,
 'results_file': <SinglefileData: uuid: 8c28521c-b669-45be-898e-446a72d50707 (pk: 4582)>}

### Outputs 
We can then check the output to ensure we are getting the correct output.
#### Geometry Optimisation

In [20]:
type(wg.outputs.results_file.value)

aiida.orm.nodes.data.singlefile.SinglefileData

We can also print the outputs of the calculation. Note the new lattice parameter is approximately 2.77 Angstroms with the `MACE-matpes-r2scan-omat-ft.model` .

In [22]:
wg.outputs.results.value.get_dict()

{'numbers': [11, 17],
 'positions': [[-0.0, 0.0, -0.0], [2.76673554, -0.0, 0.0]],
 'masses': [22.98976928, 35.45],
 'mace_forces': [[0.0, 0.0, -0.0], [-0.0, -0.0, 0.0]],
 'cell': [[0.0, 2.7667355407658, 2.7667355407658],
  [2.7667355407658, 0.0, 2.7667355407658],
  [2.7667355407658, 2.7667355407658, 0.0]],
 'pbc': [True, True, True],
 'info': {'initial_spacegroup': 'Fm-3m (225)',
  'units': {'energy': 'eV', 'forces': 'ev/Ang', 'stress': 'ev/Ang^3'},
  'final_spacegroup': 'Fm-3m (225)',
  'model': 'mlff.model',
  'arch': 'mace',
  'mace_energy': -13.771058046831,
  'mace_stress': [-0.00057726292927505,
   -0.00057726292927506,
   -0.00057726292927505,
   7.2488744797178e-18,
   3.6025047952286e-18,
   -4.1990807008407e-18],
  'system_name': 'aiida',
  'config_type': 'geom_opt'}}

#### Phonon Calculation

The results for the phonon calculation were not passed to the graph so the output can be retrieved using the alternate method. Note the results use the optimised lattice parameter for the phonon calculation.

In [23]:
wg.tasks.ph_calc.outputs.results_dict.value.get_dict()

{'phonopy': {'version': '2.47.1',
  'frequency_unit_conversion_factor': 15.633302,
  'symmetry_tolerance': 1e-05},
 'space_group': {'type': 'Fm-3m', 'number': 225, 'Hall_symbol': '-F 4 2 3'},
 'supercell_matrix': [[2, 0, 0], [0, 2, 0], [0, 0, 2]],
 'primitive_cell': {'lattice': [[0.0, 2.7667355407658, 2.7667355407658],
   [2.7667355407658, 0.0, 2.7667355407658],
   [2.7667355407658, 2.7667355407658, 0.0]],
  'points': [{'symbol': 'Na',
    'coordinates': [0.0, 0.0, 0.0],
    'mass': 22.989769},
   {'symbol': 'Cl',
    'coordinates': [0.50000000013839, 0.49999999986161, 0.49999999986161],
    'mass': 35.45}],
  'reciprocal_lattice': [[-0.18071839271693,
    0.18071839271693,
    0.18071839271693],
   [0.18071839271693, -0.18071839271693, 0.18071839271693],
   [0.18071839271693, 0.18071839271693, -0.18071839271693]]},
 'unit_cell': {'lattice': [[0.0, 2.7667355407658, 2.7667355407658],
   [2.7667355407658, 0.0, 2.7667355407658],
   [2.7667355407658, 2.7667355407658, 0.0]],
  'points': [{'